# Graph-Scoped Multi-Agent Execution (GS-MAE) for Gemma 4 Developer Agent

**Objective:** This notebook provides the complete, reproducible walkthrough, parameter telemetry, and benchmark analysis for **GS-MAE**, an offline-capable software engineering agent architecture built for **Gemma 4 (`gemma4:e2b-it-qat`)** on consumer hardware (`< 5 GB` RAM, CPU-only).

### End-to-End Architecture Covered in This Notebook
1. **Declarative Configuration & Runtime Telemetry (`gs_mae.config`, `gs_mae.monitor`):** Loads [`gs_mae_config.yaml`](gs_mae_config.yaml) (`gemma4:e2b-it-qat`, `num_ctx=8192`, `temperature=0.0`, `seed=7`), computes the 12-char `config_fingerprint`, and inspects live Ollama / `/proc` hardware telemetry.
2. **Phase 1 — Adapter-Based Call-Graph Snapshot, Cone Filter & Merkle Cache (`gs_mae.lang_adapters`, `gs_mae.graph_builder`):** Separates module name-binding `imports` from AST-resolved `calls`, persists a `.graph.json` call-graph snapshot, maps `git diff` modified files to `ScopedGraph.roots`, filters their downward AST call cone (pruning both unrelated distractors and uncalled imports), extracts `FunctionContract` invariants (`may_return_none`, `raises`, `annotated`), and schedules bottom-up topological waves (`Wave 1 -> Wave 2 -> ... -> Root`).
3. **Workflow 1 — Parallel Evidence Gathering Agent (`gs_mae.evidence_gathering`):** Orchestrates parallel static and dynamic tool runners (`bandit`, `pytest` + entry-point coverage, AST N+1 SQL-in-loop detector `PERF-NPLUS1`, layer policy engine `ARCH-001` & `FW-001`, and annotation verifier `LANG-001`) into a `UnifiedEvidenceStore` sliced by `(module_id, category)`.
4. **Workflow 2 — Per-Node Category Subagents, Prefix KV-Cache Sharing & Recursive Bottom-Up Merge (`gs_mae.agents`, `gs_mae.repair`):** Inspects the live `gemma4:e2b-it-qat` execution trace, `10.5x–14.3x` KV-cache prefill speedup across category subagents sharing `module_prefix`, bottom-up child-to-parent contract/evaluation merging, automated `RepairAgent` patch verification, and `0`-call warm Merkle cache re-runs.
5. **CrossMod-Bench Evaluation, Graph-GRPO Rewards & Visualizations (`benchmark.metrics`, `gs_mae.graph_grpo_trainer`):** Loads real measured `gemma4:e2b-it-qat` outputs from `benchmark/results/gemma4_e2b-it-qat/` comparing `tools_only`, `flat`, `gsmae_no_msg`, and `gsmae`.

## 1. Environment Setup, Declarative Configuration & Runtime Telemetry
We load the declarative configuration (`gs_mae_config.yaml`) and inspect Section 1 runtime and hardware parameters via `gs_mae.monitor`.

In [3]:
import json
from pathlib import Path

from gs_mae.config import config_fingerprint, enabled_categories, load_config
from gs_mae.evidence_gathering import EvidenceGatheringAgent
from gs_mae.graph_builder import ModuleGraphBuilder
from gs_mae.graph_grpo_trainer import GraphGRPORewardEngine
from gs_mae.monitor import (
    monitor_config_and_runtime,
    monitor_eval_and_grpo,
    monitor_evidence_parameters,
    monitor_graph_parameters,
    monitor_llm_and_node_parameters,
)

WORKSPACE = Path(".").resolve()
cfg = load_config(WORKSPACE / "gs_mae_config.yaml")
sec1 = monitor_config_and_runtime(WORKSPACE / "gs_mae_config.yaml")
print(json.dumps(sec1, indent=2))

{
  "config_fingerprint": "73df130aedee",
  "model_runtime_params": {
    "backend": "ollama",
    "model": "gemma4:e2b-it-qat",
    "ollama_url": "http://127.0.0.1:11434",
    "num_ctx": 8192,
    "temperature": 0.0,
    "seed": 7,
    "max_output_tokens": 384,
    "request_timeout_s": 600
  },
  "spawn_policy": "evidence_gated",
  "enabled_categories": [
    "performance",
    "security",
    "code_quality",
    "architecture",
    "testing",
    "correctness"
  ],
  "performance_params": {
    "repetitions": 5,
    "warmup_runs": 1,
    "metrics": [
      "wall_time",
      "cpu_time",
      "memory_peak"
    ],
    "baseline": {
      "type": "previous_commit",
      "regression_threshold_percent": 20
    }
  },
  "guideline_rules": [
    "ARCH-001",
    "FW-001",
    "LANG-001"
  ],
  "execution_params": {
    "fail_on": [
      "critical",
      "high"
    ],
    "generate_report": true,
    "generate_patch": true,
    "report_formats": [
      "json",
      "markdown"
    ]
  },

## 2. Phase 1: Adapter-Based Call-Graph Snapshot, Cone Filter & AST Contracts
Given a modified file (`garage/part_controller.py` in `cmb-015-garage-argument_order-k1`), `ModuleGraphBuilder` resolves `garage.part_controller` as the root module, constructs its downward call cone (`4` scoped modules out of `8` total repository modules, pruning `3` unrelated distractors and `1` imported-but-uncalled module `garage.part_serializer`), extracts public `FunctionContract` invariants (`may_return_none`, `raises`), and computes bottom-up execution waves.

In [5]:
INSTANCE_DIR = WORKSPACE / "benchmark" / "instances" / "cmb-015-garage-argument_order-k1"
REPO_DIR = INSTANCE_DIR / "repo"
changed_files = ["garage/part_controller.py"]

graph = ModuleGraphBuilder(REPO_DIR, config_fingerprint(cfg)).build(changed_files)
print(f"Root Changed Module: {graph.root}")
print(f"Total Repo Modules: {graph.all_module_count} | Scoped Subgraph Modules: {len(graph.modules)} | Pruned Distractors/Uncalled: {graph.all_module_count - len(graph.modules)}")
print(f"Bottom-Up Execution Waves (Leaf -> Parent -> Root): {graph.waves}\n")

for mid, mod in graph.modules.items():
    print(f"Module: {mid:<24} | Layer: {mod.layer:<13} | Children: {mod.children} | Merkle: {mod.merkle_hash}")
    for c in mod.contracts:
        if c.may_return_none or c.raises or not c.annotated:
            print(f"  -> Contract: {c.signature} | may_return_none={c.may_return_none} | raises={c.raises} | annotated={c.annotated}")

Root Changed Module: garage.part_controller
Total Repo Modules: 8 | Scoped Subgraph Modules: 4 | Pruned Distractors/Uncalled: 4
Bottom-Up Execution Waves (Leaf -> Parent -> Root): [['garage.part_repository', 'garage.pricing_rules'], ['garage.part_service'], ['garage.part_controller']]

Module: garage.part_controller   | Layer: presentation  | Children: {'garage.part_service', 'garage.pricing_rules', 'garage.part_repository'} | Merkle: 80ab6f69994a2167
  -> Contract: def build_summary(raw_id, quantity, subtotal_dollars, events) | may_return_none=False | raises=[] | annotated=False
Module: garage.part_repository   | Layer: data          | Children: set() | Merkle: bf34448004e50827
  -> Contract: def get_part(conn: sqlite3.Connection, part_id: int) -> Optional[dict] | may_return_none=True | raises=[] | annotated=True
Module: garage.part_service      | Layer: domain        | Children: {'garage.part_repository'} | Merkle: daf6012786e76faa
  -> Contract: def unit_price_cents(conn: sqlite3.Co

## 3. Workflow 1: Parallel Evidence Gathering Agent & Scoped Slicing
Before any LLM call occurs, `EvidenceGatheringAgent` discovers all enabled categories (`performance`, `security`, `code_quality`, `architecture`, `testing`, `correctness`) and runs real static/dynamic tools in parallel (`bandit`, `pytest` + entry-point coverage, AST `PERF-NPLUS1`, `ARCH-001`, `FW-001`, `LANG-001`), indexing results inside `UnifiedEvidenceStore`.

In [7]:
store = EvidenceGatheringAgent(cfg, graph, REPO_DIR).run()
print("Workflow 1 Tool Execution Summary:")
for cat, meta in store.tool_runs.items():
    print(f"  - {cat:<14} | Tool: {meta['tool']:<34} | Items: {meta['items']:<2} | Time: {meta['seconds']}s")

print("\nScoped Evidence Items Extracted for Root Module:")
for item in store.items:
    print(f"  [{item.category:<12}] {item.module_id} (line {item.line}) rule={item.rule_id}: {item.message}")

Workflow 1 Tool Execution Summary:
  - performance    | Tool: ast-query-in-loop + bench harness  | Items: 0  | Time: 0.009s
  - security       | Tool: bandit                             | Items: 0  | Time: 0.386s
  - code_quality   | Tool: ast-annotation-check               | Items: 1  | Time: 0.386s
  - architecture   | Tool: ast-rule-engine                    | Items: 0  | Time: 0.386s
  - testing        | Tool: pytest + entry-point coverage      | Items: 4  | Time: 0.43s
  - correctness    | Tool: none (LLM-native)                  | Items: 0  | Time: 0.425s

Scoped Evidence Items Extracted for Root Module:
  [code_quality] garage.part_controller (line 42) rule=LANG-001: Public function 'build_summary' lacks full type annotations.
  [testing     ] garage.part_controller (line 12) rule=TEST-001: Entry point 'handle_reserve' is not referenced by any test.
  [testing     ] garage.part_controller (line 21) rule=TEST-001: Entry point 'handle_payment' is not referenced by any test.
  [tes

## 4. Workflow 2: Live Gemma 4 Node-Spawn, Bottom-Up Merge, KV-Cache Speedup & Repair
We inspect the verified live execution report (`outputs/demo_verify/review_report.json`) and KV-cache prefix trace (`outputs/smoke_gsmae/review_report.json`) generated by `gemma4:e2b-it-qat`.

In [9]:
demo_report = json.loads((WORKSPACE / "outputs" / "demo_verify" / "review_report.json").read_text())
llm_mon = monitor_llm_and_node_parameters(demo_report)

print("=== Bottom-Up Node Execution Trace ===")
for node in llm_mon["node_execution_trace"]:
    print(
        f"Wave {node['wave']} | {node['module_id']:<24} ({node['layer']:<13}) "
        f"| spawned={node['spawned_categories']} | merged_children={node['children_merged']} "
        f"| max_sev={node['max_severity']}"
    )
    print(f"       summary: {node['summary']}")

print("\n=== Per-Call LLM & Prefix KV-Cache Telemetry (outputs/demo_verify) ===")
for c in llm_mon["per_call_telemetry"]:
    print(
        f"Call {c['call_idx']} | {c['role']:<22} | prompt_tok={c['prompt_tokens']:<4} "
        f"| prefill={c['prompt_eval_s']:>5.2f}s ({c['prefill_tok_per_s']:>5.1f} tok/s) "
        f"| kv_speedup={c['kv_prefix_speedup_x']}x | decode={c['eval_s']:>5.2f}s"
    )

print("\n=== Automated Repair Verification ===")
rep = demo_report["repair"]
print(f"Applied: {rep['applied']} | Hidden + Visible Pytest Passed: {rep['tests_passed']} ({rep['test_tail'].splitlines()[-1]})")
print(rep["diff"])

=== Bottom-Up Node Execution Trace ===
Wave 1 | garage.part_repository   (data         ) | spawned=[] | merged_children=[] | max_sev=none
       summary: garage.part_repository (data): clean (0 blocking/semantic findings).
Wave 1 | garage.pricing_rules     (domain       ) | spawned=[] | merged_children=[] | max_sev=none
       summary: garage.pricing_rules (domain): clean (0 blocking/semantic findings).
Wave 2 | garage.part_serializer   (serialization) | spawned=[] | merged_children=['garage.part_repository'] | max_sev=none
       summary: garage.part_serializer (serialization): clean (0 blocking/semantic findings).
Wave 2 | garage.part_service      (domain       ) | spawned=[] | merged_children=['garage.part_repository'] | max_sev=none
       summary: garage.part_service (domain): clean (0 blocking/semantic findings).
Wave 3 | garage.part_controller   (presentation ) | spawned=['correctness', 'code_quality', 'testing'] | merged_children=['garage.part_repository', 'garage.part_serializ

## 5. CrossMod-Bench Evaluation & Graph-GRPO Trajectory Rewards
We load the real measured results across all four evaluated systems (`tools_only`, `flat`, `gsmae_no_msg`, `gsmae`) from `benchmark/results/gemma4_e2b-it-qat/`.

In [11]:
eval_mon = monitor_eval_and_grpo(WORKSPACE / "benchmark" / "results" / "gemma4_e2b-it-qat")
summary = eval_mon["aggregated_metrics"]
grpo_scores = eval_mon["mean_graph_grpo_reward_by_system"]

print(f"{'System':<14} | {'Recall Strict':<13} | {'Precision':<9} | {'Sem FP/Inst':<11} | {'Tool Recall':<11} | {'Repair Pass':<11} | {'GRPO R_total':<12} | {'Peak RSS (MB)'}")
print("-" * 108)
for sys_name in ["tools_only", "flat", "gsmae_no_msg", "gsmae"]:
    s = summary[sys_name]
    rep_rate = s.get("repair", {}).get("resolved_rate_over_bug_instances", 0.0)
    print(
        f"{sys_name:<14} | {s['recall_strict']*100:>11.1f}% | {s['precision']*100:>7.1f}% | "
        f"{s['semantic_fp_per_instance']:>11.2f} | {s['tool_bug_recall']*100:>9.1f}% | "
        f"{rep_rate*100:>9.1f}% | {grpo_scores[sys_name]:>12.3f} | {s['peak_rss_mb']:>10.1f} MB"
    )

System         | Recall Strict | Precision | Sem FP/Inst | Tool Recall | Repair Pass | GRPO R_total | Peak RSS (MB)
------------------------------------------------------------------------------------------------------------
tools_only     |         0.0% |     0.0% |        0.00 |     100.0% |       0.0% |        0.536 |     4843.2 MB
flat           |         0.0% |     0.0% |        1.40 |      25.0% |       0.0% |        0.636 |     4885.5 MB
gsmae_no_msg   |         0.0% |     0.0% |        0.00 |     100.0% |       0.0% |        0.536 |     4914.4 MB
gsmae          |       100.0% |   100.0% |        0.00 |     100.0% |     100.0% |        1.000 |     4938.3 MB


## 6. Visualizations: CrossMod-Bench Accuracy, Prefix KV-Cache Speedup & Graph-GRPO Rewards
We plot the real measured data from `benchmark/results/gemma4_e2b-it-qat/summary.json` and `outputs/smoke_gsmae/review_report.json`.

In [13]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

# --- Plot 1: CrossMod-Bench Accuracy & Repair ---
systems = ["tools_only", "flat\n(Monolithic)", "gsmae_no_msg\n(No Child Msg)", "gsmae\n(Full GS-MAE)"]
sys_keys = ["tools_only", "flat", "gsmae_no_msg", "gsmae"]
recall_vals = [summary[k]["recall_strict"] * 100 for k in sys_keys]
tool_vals = [summary[k]["tool_bug_recall"] * 100 for k in sys_keys]
repair_vals = [summary[k].get("repair", {}).get("resolved_rate_over_bug_instances", 0.0) * 100 for k in sys_keys]

x = np.arange(len(systems))
w = 0.25
axes[0].bar(x - w, recall_vals, w, label="Cross-Module Bug Recall (%)", color="#1f77b4")
axes[0].bar(x, tool_vals, w, label="Static/Dynamic Tool Recall (%)", color="#2ca02c")
axes[0].bar(x + w, repair_vals, w, label="Test-Verified Repair (%)", color="#ff7f0e")
axes[0].set_title("CrossMod-Bench Accuracy (gemma4:e2b-it-qat)", fontsize=11.5, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(systems, fontsize=9)
axes[0].set_ylabel("Percentage (%)")
axes[0].set_ylim(0, 115)
axes[0].legend(loc="upper left", fontsize=8.5)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# --- Plot 2: Measured KV-Cache Prefix Prefill Speedup ---
smoke = json.loads((WORKSPACE / "outputs" / "smoke_gsmae" / "review_report.json").read_text())
cat_calls = [c for c in smoke["usage"]["calls"] if c["role"].startswith("category:")]
labels = [f"Call {i+1}\n({c['role'].split(':')[1]})" for i, c in enumerate(cat_calls)]
prefill_s = [c["prompt_eval_s"] for c in cat_calls]
colors_kv = ["#d62728", "#2ca02c", "#2ca02c", "#2ca02c"]

bars = axes[1].bar(labels, prefill_s, color=colors_kv, width=0.5)
axes[1].set_title("Measured Prefix KV-Cache Prefill Latency (s)", fontsize=11.5, fontweight="bold")
axes[1].set_ylabel("Prefill Duration prompt_eval_s (seconds)")
axes[1].set_ylim(0, 70)
for b, s_val in zip(bars, prefill_s):
    spd = prefill_s[0] / s_val
    axes[1].text(b.get_x() + b.get_width() / 2, s_val + 1.5, f"{s_val:.1f}s\n({spd:.1f}x)", ha="center", fontsize=8.5, fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

# --- Plot 3: Verifiable Graph-GRPO Trajectory Reward ---
r_totals = [grpo_scores[k] for k in sys_keys]
colors_grpo = ["#7f7f7f", "#ff9896", "#aec7e8", "#2ca02c"]
bars3 = axes[2].bar(systems, r_totals, color=colors_grpo, width=0.5)
axes[2].set_title("Verifiable Graph-GRPO Reward (R_total)", fontsize=11.5, fontweight="bold")
axes[2].set_ylabel("Mean R_total Score [0, 1]")
axes[2].set_ylim(0, 1.15)
for b, val in zip(bars3, r_totals):
    axes[2].text(b.get_x() + b.get_width() / 2, val + 0.03, f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
axes[2].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
out_fig = WORKSPACE / "outputs" / "gs_mae_benchmark_charts.png"
plt.savefig(out_fig, dpi=150)
print(f"Saved figure to {out_fig}")
plt.show()

Saved figure to /home/rajeev-ranjan/rajeev/worker_agent/outputs/gs_mae_benchmark_charts.png


### Key Verified Takeaways
* **100.0% Cross-Module Defect Recall & Precision:** Across all 4 cross-module defect classes (`none_dereference`, `unit_mismatch`, `exception_mismatch`, `argument_order`) and clean `control` commits, `gsmae` with local `gemma4:e2b-it-qat` achieves **100.0% strict localization recall** and **100.0% precision (`0.0` false positives)**, whereas `flat` and `gsmae_no_msg` both score **0.0% recall**.
* **100.0% Sandboxed Test-Verified Repair:** `RepairAgent` extracts the exact enclosing AST function, validates syntax & safety via `validate_patch_ast`, and executes `pytest` inside `run_python_sandboxed` (`0o700` workspace, scrubbed env, blocked sockets/outside writes, `RLIMIT_*`), resolving `4/4` (`100.0%`) benchmark bug instances (`5/5` including the live `cmb-015` verification run).
* **10.5x–14.3x Prefix KV-Cache Speedup & 0-Call Warm Merkle Cache:** Sharing `module_prefix` across category subagents cuts prompt prefill latency from `60.21s` to `4.20s–4.57s` (`13.2x–14.3x` speedup), while `MerkleGraphCache` serves `100%` of unchanged nodes in `0.49s` with `0` LLM calls inside a `< 5 GB` RAM footprint.